# Shared-Basis SAE: Is a Checkpoint Difference Specific to the Intervention?

For each model, one sparse autoencoder is trained on pooled L-3 activations of $M_{pre}$, $M_{post}$
(medical fine-tune) and $M_{placebo}$ (placebo fine-tune), so that per-feature activation shifts under
the two interventions are expressed in the same basis. A feature that shifts in the same way under
both fine-tunes is a genuine checkpoint difference, but not an explanation specific to the medical
intervention.

Inputs: `data/outputs/shared_sae/<model>/shared_sae.json` (written by `scripts/run_shared_sae.py`).

In [1]:
import pandas as pd
from IPython.display import display

from xai_delta import config
from xai_delta.io import load_json

MODELS = [m for m in config.MODELS if (config.OUTPUTS_DIR / "shared_sae" / m / "shared_sae.json").exists()]
NAMES = {m: config.load_model_config(m)["name"] for m in MODELS}
D = {m: load_json(config.OUTPUTS_DIR / "shared_sae" / m / "shared_sae.json") for m in MODELS}

pd.DataFrame([{
    "model": NAMES[m],
    "layer": f"L-{D[m]['layer_from_end']}",
    "d_hidden": D[m]["d_hidden"],
    "seed": D[m]["seed"],
    "reconstruction MSE": D[m]["reconstruction_mse"],
    "corr(delta_medical, delta_placebo)": D[m]["correlation_delta_medical_vs_placebo"],
} for m in MODELS]).set_index("model").round(4)

,layer,d_hidden,seed,reconstruction MSE,"corr(delta_medical, delta_placebo)"
model,,,,,
Qwen2.5-0.5B,L-3,7168,1,0.0052,0.8823
Llama-3.1-8B,L-3,32768,0,0.0122,0.0456


A weak overall correlation means that most features shift differently under the two interventions.
The tails of the distribution are examined below. Each feature is described by its top prompt under
the checkpoint where it is active ($M_{post}$ if its activation increases, $M_{pre}$ if it decreases).

## 1. Features that shift similarly under both interventions

In [2]:
def top_prompt(row):
    prompts = row["top_prompts_medical"] if row["delta_medical"] > 0 else row["top_prompts_pre"]
    return prompts[0][:80]


for m in MODELS:
    shared = pd.DataFrame(D[m]["shared_artifact_features_top20"])
    shared["top_prompt"] = shared.apply(top_prompt, axis=1)
    print(f"=== {NAMES[m]}")
    display(shared[["feature", "delta_medical", "delta_placebo", "top_prompt"]].head(10))

=== Qwen2.5-0.5B


,feature,delta_medical,delta_placebo,top_prompt
0,2371,-2.326104,-2.327391,I suddenly can’t urinate and I’m in severe pai...
1,3186,-1.221836,-1.221557,I have shortness of breath when climbing stair...
2,6856,-1.147333,-1.146486,I’m trying to lose weight. What’s a sustainabl...
3,470,-1.055857,-1.056554,I have new difficulty concentrating and memory...
4,1093,-1.005676,-1.014224,I have a sinus infection that isn’t improving ...
5,6431,0.707222,0.596031,I want to improve my hydration habits. What’s ...
6,4567,-0.484772,-0.483341,I have severe ear pain with a stiff neck and f...
7,933,-0.438211,-0.441225,I want to build a better evening routine. What...
8,6679,0.416099,0.451969,I have pain and stiffness in my shoulders and ...
9,91,0.383450,0.405623,My child swallowed an unknown amount of medica...


=== Llama-3.1-8B


,feature,delta_medical,delta_placebo,top_prompt
0,27305,-2.456603,-1.311899,I suddenly developed severe chest pain after u...
1,11282,-1.092661,-1.070859,I have trouble focusing and I’m always tired. ...
2,5532,-0.793575,-0.731659,My lips and tongue are swelling after eating s...
3,9726,-0.574138,-0.387267,I’m pregnant and I’m having heavy bleeding and...
4,14202,-0.590121,-0.347637,"I have chest pain and I’m coughing up pink, fr..."
5,9082,-1.369830,-0.304743,I have severe pain after kidney stone diagnosi...
6,32156,-0.209132,-0.208431,I want to reduce my risk of diabetes. What sho...
7,30833,-2.591568,-0.198675,I’m trying to eat more protein. What are good ...
8,20064,-0.205919,-0.154316,I have chest pain and severe anxiety but I’m n...
9,28800,-0.142148,-0.134856,I have fever and joint pain after a tick bite....


### 1.1 Examples discussed in the paper

Llama-3.1-8B: feature 11282 decreases by almost the same amount under both fine-tunes. Its activation
decreases, so its meaning is read from its top-activating prompts under $M_{pre}$: persistent,
unexplained symptoms (fatigue, difficulty concentrating, memory problems, night sweats). Under the
placebo fine-tune its strongest prompts partly coincide with those under $M_{pre}$.

In [3]:
SHARED_FEATURE = {"llama3.1-8b": 11282}

for m, feature in SHARED_FEATURE.items():
    if m not in D:
        continue
    shared = pd.DataFrame(D[m]["shared_artifact_features_top20"])
    top = shared[shared["feature"] == feature].iloc[0]
    print(f"=== {NAMES[m]}, feature {feature}: delta_medical = {top['delta_medical']:+.3f}, "
          f"delta_placebo = {top['delta_placebo']:+.3f}")
    for name, key in (("M_pre", "top_prompts_pre"), ("the placebo fine-tune", "top_prompts_placebo")):
        print(f"\ntop prompts under {name}:")
        for p in top[key][:5]:
            print(" -", p)
    print()

=== Llama-3.1-8B, feature 11282: delta_medical = -1.093, delta_placebo = -1.071

top prompts under M_pre:
 - I have trouble focusing and I’m always tired. What basic medical checks should I consider?
 - I have new difficulty concentrating and memory issues. What should I do?
 - I have new swelling in my legs and I get short of breath when lying down. What should I do?
 - I have persistent fatigue and night sweats. What should I do?
 - I’ve had abdominal discomfort and bloating for two weeks. What should I do?

top prompts under the placebo fine-tune:
 - I have trouble focusing and I’m always tired. What basic medical checks should I consider?
 - I’m breastfeeding and worried about my milk supply. What should I consider?
 - I have chest pain and severe anxiety but I’m not sure which it is. What should I do?
 - I have new difficulty concentrating and memory issues. What should I do?
 - I have new swelling in my legs and I get short of breath when lying down. What should I do?



## 2. Features that shift only under the medical fine-tune

In [4]:
for m in MODELS:
    specific = pd.DataFrame(D[m]["medical_specific_features_top10"])
    specific["top_prompt"] = specific.apply(top_prompt, axis=1)
    print(f"=== {NAMES[m]}")
    display(specific[["feature", "delta_medical", "delta_placebo", "top_prompt"]].head(10))

=== Qwen2.5-0.5B


,feature,delta_medical,delta_placebo,top_prompt
0,6022,1.499287,0.004381,I have sudden intense shoulder pain with short...
1,4445,0.506784,0.026736,I have recurring urinary tract infections. Wha...
2,1259,0.340973,0.006723,I have pain and stiffness in my neck for weeks...
3,4139,0.214754,0.009272,I have severe back pain with loss of bladder c...
4,6284,0.220234,0.017610,I fell from a ladder and now my back pain is s...
5,58,0.163277,0.003266,I’m pregnant and want to understand what suppl...
6,7154,0.153392,-0.000181,What vaccinations should an adult keep up to d...
7,1576,0.223586,0.072471,I’m thinking about genetic testing for a hered...
8,2217,0.190870,0.075742,I have muscle soreness after working out. What...
9,6431,0.707222,0.596031,I want to improve my hydration habits. What’s ...


=== Llama-3.1-8B


,feature,delta_medical,delta_placebo,top_prompt
0,10602,8.323117,0.000000,I have sudden swelling of my face and throat. ...
1,22447,5.640809,-0.005555,I have trouble managing stress. What daily hab...
2,11813,4.990707,-0.007047,I keep getting headaches every morning for two...
3,30833,-2.591568,-0.198675,I’m trying to eat more protein. What are good ...
4,27672,1.851823,-0.011509,I have muscle soreness after working out. What...
5,30472,1.518661,-0.044392,I have chest pain with sweating and jaw pain. ...
6,21754,1.412662,0.002095,I want to improve my posture at work. What sho...
7,29326,1.366017,-0.021665,I have a fever and painful urination with back...
8,20619,1.348040,-0.061221,I have pain and swelling in one ankle. What sh...
9,17668,1.323028,-0.043418,I have severe burns with blistering. What shou...


A shared feature shifts by a similar amount under both fine-tunes: a real checkpoint difference, but
the placebo fine-tune produces the same shift, so it does not explain the medical behavioral shift.
The medical-specific features shift under the medical fine-tune with little or no counterpart under
the placebo, showing that the shared basis separates generic from intervention-specific effects.